In [1]:
# General imports
from netsim.netSimPy import *
from netsim.netSimPy.common.evaluators import EventEvaluator, SimpleEvaluator
from netsim.netSimPy.common.allocators import sap_ff, least_fragmentation_band_prioritization, Variant, shortest_route_most_available_band
from netsim.netSimPy.common.allocators import band_fragmentation_porcentage, route_fragmentation_porcentage

/Users/jbcedeno/Documents/projcts/multiband-gymnasium/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class NetworkEvaluator(EventEvaluator):
    metrics = None
    
    def __init__(self, bands=["C", "S", "L", "E"],
                filename = "net.evaluations",
                header = None,
                info_keywords = [],
                override_existing = True
            ):
        self.metrics = {
            "steps": 0,
            "blockedEvents": 0,
            "attendedByRouteIndex": {},
            "totalAttendedByBand": {},
            "totalAttended": 0,
            # No usar:
            "fragmentationByBand": {},
            # Añadir
            # Atendidos Modulación
            # Bitrate total?
            # Bitrate por banda
        }
        super().__init__(
            filename,
            header,
            info_keywords=list(self.metrics.keys()) + info_keywords,
            override_existing=override_existing)
        self.bands = bands

        self.totalAttended = 0
        for band in self.bands:
            self.metrics["totalAttendedByBand"][band] = 0
            self.metrics["fragmentationByBand"][band] = []
    
    def _on_run_end(self, args):
        block = self.metrics["blockedEvents"]
        steps = self.metrics["steps"]
        print(f"Blocking probability: {round(block/steps, 6)}")
        
    
    def _on_update(self, args):
        event: Event = args["event"]
        self.metrics["steps"] = args["steps"]
        network: Network = args.get("network", None)
        if event.getType() != EventType.Departure:
            self.metrics["blockedEvents"] += 1
        if network is not None:
            if event.getType() == EventType.Departure:
                # the event was allocated, get connection associated:
                con = network.getConnection(event.id)
                self.totalAttended += 1
                self.metrics["totalAttendedByBand"][con.getBand(con.linksID[0])] += 1
                if con.routeIndex is not None:
                    if con.routeIndex not in self.metrics["attendedByRouteIndex"]:
                        self.metrics["attendedByRouteIndex"][con.routeIndex] = 1
                    else:
                        self.metrics["attendedByRouteIndex"][con.routeIndex] += 1
            for band in self.bands:
                self.metrics["fragmentationByBand"][band].append(
                    network.getBandFragmentation(band)
                )
        return self.metrics

In [6]:
M_LAMBDA = 200000
network = Network(
    networkFileName =  "../../networks/nsfnet/network.json",
    pathsFileName= "../../networks/nsfnet/routes.json",
    bitrateFilename= "../../networks/nsfnet/bitrates_4_bands.json",
)
generator = EventsGenerator(mLambda=M_LAMBDA)

sim_args = dict(
    eventsGenerator=generator,
    network = network,
    allocator = route_fragmentation_porcentage(3, Variant.Least_Fragmentation),
    # allocator = sap_ff(3)
)
simple_callback = SimpleEvaluator()
simulator = NetworkSimulator(**sim_args)
simulator.run(500000, simple_callback)
# callback = NetworkEvaluator()
# simulator.run(5000, callback)
# simulator.run(5000)


Total blocked events: 4432


In [ ]:
M_LAMBDA = 200000
network = Network(
    networkFileName =  "../../networks/nsfnet/network.json",
    pathsFileName= "../../networks/nsfnet/routes.json",
    bitrateFilename= "../../networks/nsfnet/bitrates_4_bands.json",
)
generator2 = EventsGenerator(mLambda=M_LAMBDA)

sim_args2 = dict(
    network = network,
    eventsGenerator=generator2,
    allocator = band_fragmentation_porcentage(3, Variant.Greater_Fragmentation),
)

simulator2 = NetworkSimulator(**sim_args2)



In [ ]:
results = {
    "Sim1": [],
    "Sim2": []
}

for traffic in [ 150000, 200000, 250000, 300000, 350000,400000, 450000, 500000, 550000, 600000, 650000, 700000, 100000]:
    print(f"{traffic}-Sim1")
    simulator.reset()
    simulator.setLambda(traffic)
    results["Sim1"].append(simulator.run(1000000))
    print(f"{traffic}-Sim2")
    simulator2.reset()
    simulator2.setLambda(traffic)
    results["Sim2"].append(simulator2.run(1000000))


